In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/voe_bem/bronze/raw_files"

def ingest_bronze(read_path, table_name):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")   # sem tipagem - tudo como string
        .option("sep", ";")
        .csv(read_path)
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )
    # sanitize column names for Delta compatibility
    invalid_chars = " ,;{}()\n\t="
    df = df.toDF(*[c.translate(str.maketrans(invalid_chars, "_" * len(invalid_chars))) for c in df.columns])
    # overwrite = idempotente: roda quantas vezes quiser, nunca duplica
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    print(f"{table_name}: {df.count()} linhas gravadas")

# 1. VRA - 12 arquivos mensais
ingest_bronze(f"{base_path}/VRA_*.csv", "voe_bem.bronze.vra")

# 2. Aeródromos
ingest_bronze(f"{base_path}/AerodromosPublicos.csv", "voe_bem.bronze.aerodromos")

# 3. Empresas nacionais
ingest_bronze(f"{base_path}/pda_empresas_aereas_nacionais.csv", "voe_bem.bronze.empresas_nacionais")

# 4. Empresas estrangeiras
ingest_bronze(f"{base_path}/pda_empresas_aereas_estrangeiros.csv", "voe_bem.bronze.empresas_estrangeiras")